## 📝 Chap04-2. Summary
- gpt-5.6-luna 을 활용해 PDF파일을 요약해 MD파일로 만들어보자.

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import pymupdf

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

def pdf_to_text(pdf_file_path: str):
    doc = pymupdf.open(pdf_file_path)

    header_height = 80
    footer_height = 80

    full_text = ''

    for page in doc:
        rect = page.rect # 페이지 크기 가져오기
        
        header = page.get_text(clip=(0, 0, rect.width , header_height))
        footer = page.get_text(clip=(0, rect.height - footer_height, rect.width , rect.height))
        text = page.get_text(clip=(0, header_height, rect.width , rect.height - footer_height))

        full_text += text + '\n------------------------------------\n'

    # 파일명만 추출
    pdf_file_name = os.path.basename(pdf_file_path)
    pdf_file_name = os.path.splitext(pdf_file_name)[0] # 확장자 제거

    txt_file_path = f'output/{pdf_file_name}_with_preprocessing.txt'

    with open(txt_file_path, 'w', encoding='utf-8') as f:
        f.write(full_text)

    return txt_file_path


def summarize_txt(file_path: str): # ①
    client = OpenAI(api_key=api_key)

    # ② 주어진 텍스트 파일을 읽어들인다.
    with open(file_path, 'r', encoding='utf-8') as f:
        txt = f.read()

    # ③ 요약을 위한 시스템 프롬프트를 생성한다.
    system_prompt = f'''
    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고, 주요 내용을 요약하라. 

    작성해야 하는 포맷은 다음과 같다. 
    
    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)
    
    ## 저자 소개

    
    =============== 이하 텍스트 ===============

    { txt }
    '''

    print(system_prompt)
    print('=========================================')

    # ④ OpenAI API를 사용하여 요약을 생성한다.
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": system_prompt},
        ]
    )

    return response.choices[0].message.content

def summarize_pdf(pdf_file_path: str, output_file_path: str):
    txt_file_path = pdf_to_text(pdf_file_path)
    summary = summarize_txt(txt_file_path)

    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(summary)

    return summary


if __name__ == '__main__':
    pdf_file_path = "data/ImpactOfShortForm.pdf"
    summary = summarize_pdf(pdf_file_path, 'output/crop_model_summary2.md')


    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고, 주요 내용을 요약하라. 

    작성해야 하는 포맷은 다음과 같다. 

    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개


    =============== 이하 텍스트 ===============

    Study on University Students’ Attitudes, Motivations for Use, 
and the Impact of Short-Form Content on Mental Health 
심층 인터뷰를 통한 숏폼 콘텐츠에 대한 대학생들의 
태도, 이용동기 및 정신건강에 미치는 영향 연구 
Namhyun Um1 
엄남현1 
1 Professor, School of Advertising & Public Relations, Hongik University, Korea, 
goldmund@hongik.ac.kr 
 
 
Abstract: Excessive use of short-form content can lead to psychological, cognitive, and physical issues 
for users, and recent studies report that such problems are particularly pronounced among young adults, 
especially university students who are in a transitional phase of identity formation and academic 
development. In response, this study explores how excessive engagement with short-form video content 
shapes the psychological and behavioral well-being of university students, focusing 

In [8]:
print(summary)

# 심층 인터뷰를 통한 숏폼 콘텐츠에 대한 대학생들의 태도, 이용동기 및 정신건강에 미치는 영향 연구

## 저자의 문제 인식 및 주장 (15문장 이내)

숏폼 콘텐츠는 짧은 시간에 강한 자극과 즉각적인 만족을 제공하고 알고리즘과 자동재생을 통해 반복 시청을 유도한다.  
대학생들은 숏폼을 재미와 정보 습득뿐 아니라 스트레스 해소, 감정적 위로, 현실 회피, 사회적 유행과 소속감 유지를 위해 이용한다.  
그러나 이러한 이용이 반복되면 단순한 여가 활동을 넘어 중독적 소비로 발전할 수 있다.  
본 연구는 숏폼 콘텐츠의 과도한 이용이 대학생의 정신건강과 자기조절 능력, 학업 및 일상생활에 미치는 영향을 탐색하는 데 목적이 있다.  
연구자는 부정적 영향을 경험한 대학생 12명을 대상으로 심층 인터뷰를 실시하고 주제 분석을 진행했다.  
분석 결과, 반복적이고 충동적인 시청은 불안, 무기력, 공허함, 죄책감, 자기비난 등의 정서적 반응을 유발했다.  
이러한 정서적 어려움은 다시 숏폼 콘텐츠를 찾게 만들고 자기조절 능력을 약화시키는 악순환으로 이어졌다.  
참여자들은 시청 시간 통제에 실패하면서 수면 부족, 피로, 집중력 저하, 생활 리듬 붕괴 등을 경험했다.  
특히 과제 지연, 강의 집중력 저하, 시험 준비 부족, 학업 성취 및 자기효능감 저하와 같은 부정적 결과가 나타났다.  
가족이나 친구와 함께 있는 상황에서도 숏폼을 시청하면서 대화와 대면 소통이 줄고 인간관계의 질이 낮아지는 사례도 확인되었다.  
앱 삭제, 시간 제한, 차단 프로그램, 디지털 디톡스 등 개인적 통제 전략은 일시적인 효과에 그치는 경우가 많았다.  
이는 숏폼 중독이 개인의 의지 부족만이 아니라 알고리즘 구조, 사회적 연결 욕구, FOMO, 정서적 취약성이 결합한 문제임을 보여준다.  
저자는 숏폼 과잉 이용이 정서적 반응을 매개로 자기조절 실패를 일으키고, 이를 통해 학업·일상·대인관계의 기능 저하로 이어진다고 주장한다.  
따라서 해결을 위해서는 디지털 리터러시 교육, 정서 조절 